Title: Topic Modeling using LDA, LSA, NMF on BBC News Dataset

Objective: To compare the performance of LDA, LSA and NMF techniques for extracting hidden topics from news articles  

In [1]:
# Install libraries

In [2]:
!pip install pandas numpy nltk scikit-learn

In [3]:
# Import libraries

In [4]:
import pandas as pd
import nltk
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [5]:
# Load Dataset

In [18]:
import pandas as pd

df = pd.read_csv('/content/bbc-news-data.csv', sep='\t')

print(df.columns)

Index(['category', 'filename', 'title', 'content'], dtype='object')


In [19]:
df.head()

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [31]:
# Data cleaning
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    words = text.split()

    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

In [23]:
df['clean_text'] = df['content'].apply(clean_text)

In [24]:
df[['content','clean_text']].head()

,content,clean_text
0,Quarterly profits at US media giant TimeWarne...,quarterly profits us media giant timewarner ju...
1,The dollar has hit its highest level against ...,dollar hit highest level euro almost three mon...
2,The owners of embattled Russian oil giant Yuk...,owners embattled russian oil giant yukos ask b...
3,British Airways has blamed high fuel prices f...,british airways blamed high fuel prices drop p...
4,Shares in UK drinks and food firm Allied Dome...,shares uk drinks food firm allied domecq risen...


In [32]:
# LDA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

cv = CountVectorizer(max_features=1000)

X = cv.fit_transform(df['clean_text'])

lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

lda.fit(X)

LatentDirichletAllocation(n_components=5, random_state=42)

In [26]:
feature_names = cv.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    print(f"\nTopic {topic_idx+1}")

    print([feature_names[i]
           for i in topic.argsort()[-10:]])


Topic 1
['could', 'mr', 'also', 'one', 'new', 'mobile', 'technology', 'music', 'people', 'said']

Topic 2
['company', 'last', 'new', 'best', 'also', 'film', 'bn', 'us', 'year', 'said']

Topic 3
['minister', 'blair', 'party', 'election', 'people', 'labour', 'government', 'would', 'mr', 'said']

Topic 4
['first', 'league', 'half', 'time', 'chelsea', 'would', 'united', 'game', 'said', 'club']

Topic 5
['last', 'time', 'win', 'england', 'one', 'first', 'game', 'year', 'world', 'said']


In [33]:
# LSA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

tfidf = TfidfVectorizer(max_features=1000)

X_tfidf = tfidf.fit_transform(df['clean_text'])

lsa = TruncatedSVD(
    n_components=5,
    random_state=42
)

lsa.fit(X_tfidf)

TruncatedSVD(n_components=5, random_state=42)

In [28]:
terms = tfidf.get_feature_names_out()

for i, comp in enumerate(lsa.components_):
    print(f"\nTopic {i+1}")

    terms_comp = zip(terms, comp)

    sorted_terms = sorted(
        terms_comp,
        key=lambda x: x[1],
        reverse=True
    )

    for term, weight in sorted_terms[:10]:
        print(term)


Topic 1
said
mr
would
year
us
people
new
also
one
government

Topic 2
mr
labour
election
blair
government
party
brown
minister
tax
bn

Topic 3
labour
mr
blair
election
party
england
brown
game
win
howard

Topic 4
film
best
awards
award
actor
actress
festival
films
oscar
director

Topic 5
bn
economy
film
growth
year
best
bank
oil
economic
dollar


In [34]:
# NMF
from sklearn.decomposition import NMF

nmf = NMF(
    n_components=5,
    random_state=42
)

W = nmf.fit_transform(X_tfidf)

H = nmf.components_

In [30]:
feature_names = tfidf.get_feature_names_out()

for topic_idx, topic in enumerate(H):

    print(f"\nTopic {topic_idx+1}")

    top_words = [feature_names[i]
                 for i in topic.argsort()[-10:]]

    print(top_words)


Topic 1
['bank', 'economy', 'company', 'market', 'sales', 'year', 'growth', 'us', 'said', 'bn']

Topic 2
['cup', 'wales', 'team', 'play', 'first', 'said', 'match', 'win', 'england', 'game']

Topic 3
['minister', 'would', 'government', 'brown', 'party', 'said', 'election', 'blair', 'labour', 'mr']

Topic 4
['director', 'oscar', 'festival', 'films', 'actress', 'actor', 'award', 'awards', 'best', 'film']

Topic 5
['use', 'software', 'phone', 'digital', 'users', 'technology', 'music', 'said', 'mobile', 'people']


conclusion: This project implemented three topic modeling techniques: LDA, LSA and NMF on the BBC News Dataset. After preprocessing the text data, the models successfully identified hidden topics related to Business, Sports, Politics, Entertainment and Technology. The results demonstrate that topic modeling is useful for organizing and understanding large collections of text documents.